# Laboratorio 4 — Limpieza y Preparación de Datos
**Curso:** Minería de Datos (EIN132A25)

## Objetivos
- Detectar y tratar **valores faltantes**
- Identificar y manejar **outliers**
- Convertir variables categóricas (**encoding**)
- Aplicar **normalización y estandarización**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
df.head()

## 1. Valores faltantes

### Detección

In [ ]:
print('Valores faltantes por columna:')
print(df.isnull().sum())
print('\nPorcentaje de faltantes:')
print((df.isnull().mean() * 100).round(2))

In [ ]:
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Mapa de valores faltantes")
plt.show()

### Estrategias de tratamiento

In [ ]:
# Imputar Age con la mediana
mediana_edad = df["Age"].median()
df["Age"] = df["Age"].fillna(mediana_edad)
print(f"Faltantes en Age: {df['Age'].isnull().sum()}")

# Imputar Embarked con la moda
moda_embarked = df["Embarked"].mode()[0]
df["Embarked"] = df["Embarked"].fillna(moda_embarked)

# Eliminar Cabin (>77% faltantes)
df = df.drop(columns=["Cabin"])

print('Valores faltantes tras tratamiento:')
print(df.isnull().sum())

## 2. Outliers — Detección con IQR

In [ ]:
sns.boxplot(x=df["Fare"])
plt.title("Distribución de Fare — detección de outliers")
plt.show()

Q1 = df["Fare"].quantile(0.25)
Q3 = df["Fare"].quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers = df[(df["Fare"] < limite_inferior) | (df["Fare"] > limite_superior)]
print(f"Outliers detectados en Fare: {len(outliers)}")

In [ ]:
# Transformación logarítmica
df["Fare_log"] = np.log1p(df["Fare"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["Fare"], bins=30, color="steelblue")
axes[0].set_title("Fare original")
axes[1].hist(df["Fare_log"], bins=30, color="salmon")
axes[1].set_title("Fare log-transformado")
plt.tight_layout()
plt.show()

## 3. Encoding de variables categóricas

### Label Encoding

In [ ]:
df["Sex_encoded"] = df["Sex"].map({"male": 0, "female": 1})
print(df[["Sex", "Sex_encoded"]].head())

### One-Hot Encoding

In [ ]:
embarked_dummies = pd.get_dummies(df["Embarked"], prefix="Embarked")
print(embarked_dummies.head())

df = pd.concat([df, embarked_dummies], axis=1)
df = df.drop(columns=["Embarked"])

## 4. Normalización y Estandarización

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

scaler_minmax = MinMaxScaler()
df[["Age_norm", "Fare_norm"]] = scaler_minmax.fit_transform(df[["Age", "Fare"]])

scaler_std = StandardScaler()
df[["Age_std", "Fare_std"]] = scaler_std.fit_transform(df[["Age", "Fare"]])

print(df[["Age", "Age_norm", "Age_std", "Fare", "Fare_norm", "Fare_std"]].describe().round(3))

## 5. Ejercicios

### Ejercicio 1 — Reporte de faltantes

In [ ]:
# TODO: cargar el dataset original y reportar faltantes por columna
df_original = pd.read_csv(url)
print(df_original.isnull().sum())

### Ejercicio 2 — Tratamiento de faltantes

In [ ]:
# TODO: imputar Age con mediana y Embarked con moda
df2 = df_original.copy()
df2["Age"] = df2["Age"].fillna(df2["Age"].median())
df2["Embarked"] = df2["Embarked"].fillna(df2["Embarked"].mode()[0])
print('Faltantes restantes:', df2[["Age", "Embarked"]].isnull().sum().sum())

### Ejercicio 3 — Escalar Age y Fare

In [ ]:
# TODO: escalar Age y Fare con StandardScaler y verificar media=0, std=1
from sklearn.preprocessing import StandardScaler
df_temp = df2.dropna(subset=["Age", "Fare"])
sc = StandardScaler()
scaled = sc.fit_transform(df_temp[["Age", "Fare"]])
print('Media:', scaled.mean(axis=0).round(6))
print('Std:', scaled.std(axis=0).round(6))

### Desafío — Pipeline de limpieza completo

In [ ]:
from sklearn.preprocessing import StandardScaler

def pipeline_limpieza(df_raw):
    df_clean = df_raw.copy()
    # Valores faltantes
    df_clean["Age"] = df_clean["Age"].fillna(df_clean["Age"].median())
    df_clean["Embarked"] = df_clean["Embarked"].fillna(df_clean["Embarked"].mode()[0])
    df_clean = df_clean.drop(columns=["Cabin", "Name", "Ticket", "PassengerId"])
    # Encoding
    df_clean["Sex"] = df_clean["Sex"].map({"male": 0, "female": 1})
    embarked_dummies = pd.get_dummies(df_clean["Embarked"], prefix="Embarked", drop_first=True)
    df_clean = pd.concat([df_clean.drop(columns=["Embarked"]), embarked_dummies], axis=1)
    # Scaling
    sc = StandardScaler()
    df_clean[["Age", "Fare"]] = sc.fit_transform(df_clean[["Age", "Fare"]])
    return df_clean

df_listo = pipeline_limpieza(pd.read_csv(url))
print('Shape final:', df_listo.shape)
print('Faltantes:', df_listo.isnull().sum().sum())
df_listo.head()